# LinkedIn Connections → Notion Leads Pipeline

**End-to-end workflow:**
1. Scrape LinkedIn connections from the last 7 days (first name + email)
2. Deduplicate against existing Notion leads
3. Push new leads into `leads_crm` Notion database

**Output per lead:** `First Name`, `Email`, `Source = LinkedIn`

**Notion API limits:** 3 req/s (we target 2.8). `batch_add_rows()` pre-caches
schema once, handles 429 retries, and reports stats.

In [ ]:
# Cell 1 — Setup & Imports
import sys, json, os
from pathlib import Path
from datetime import datetime, timezone

# Ensure workspace root is on sys.path
WS_ROOT = Path.cwd()
if not (WS_ROOT / 'shared').is_dir():
    # Walk up until we find it
    for p in [WS_ROOT] + list(WS_ROOT.parents):
        if (p / 'shared').is_dir() and (p / 'Business').is_dir():
            WS_ROOT = p
            break
sys.path.insert(0, str(WS_ROOT))
os.chdir(WS_ROOT)

from shared.env_loader import load_env_for_source
load_env_for_source()

from shared.notion_client import (
    add_row, query_db, batch_add_rows, notion_rate_limiter_stats, DATABASES,
)

SCRAPER_SCRIPT = WS_ROOT / 'Business' / 'GROWTH' / 'executions' / 'Marketing' / 'linkedin_scraper.py'
OUTPUT_DIR     = SCRAPER_SCRIPT.parent

print(f'Workspace root: {WS_ROOT}')
print(f'leads_crm DB ID: {DATABASES["leads_crm"]}')
print('Ready.')

python-dotenv could not parse statement starting at line 129
python-dotenv could not parse statement starting at line 130


Workspace root: c:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge
leads_crm DB ID: 30b652ea-6c6d-81e5-838b-c71135e14982
Ready.


## Step 1 — Scrape LinkedIn Connections (last 7 days)

Runs the scraper in a subprocess. It opens Chromium, loads saved session,
scrapes the connections page, visits each profile for Contact Info → email,
and writes a JSON file.

In [2]:
# Cell 2 — Run LinkedIn Scraper
import subprocess

DAYS = 7  # ← Change this to scrape a different window

result = subprocess.run(
    [sys.executable, str(SCRAPER_SCRIPT), '--scrape', '--days', str(DAYS)],
    capture_output=True, text=True, cwd=str(WS_ROOT),
    env={**os.environ, 'PYTHONPATH': str(WS_ROOT), 'PYTHONIOENCODING': 'utf-8'}
)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:] if result.stderr else '(none)')
    raise RuntimeError(f'Scraper exited with code {result.returncode}')

print('\n✅ Scrape complete.')

[scrape] Navigating to connections pageâ€¦
[scrape] Page loaded. Collecting connections from the last 7 day(s)â€¦
[scrape] Scroll  1:   10 collected | 10 new | 0 consecutive-old
[scrape] Scroll  2:   10 collected | 0 new | 0 consecutive-old
[scrape] Scroll  3:   10 collected | 0 new | 0 consecutive-old
[scrape] Scroll  4:   10 collected | 0 new | 0 consecutive-old
[scrape] No new connections after scrolling â€” end of list.

[enrich] Fetching emails for 10 connection(s)â€¦
  [ 1/10] Daria                wk021358@student.reading.ac.uk
  [ 2/10] David                rivasentrepeneur@gmail.com
  [ 3/10] Saad                 saadelhaddadi.5@gmail.com
  [ 4/10] Aman                 shaikhaman0207@gmail.com
  [ 5/10] Denis                denis.desclos@gmail.com
  [ 6/10] Oluwaseun            kykuit_merchandise@yahoo.com
  [ 7/10] Arnav                arnav.s2207@gmail.com
  [ 8/10] Tyler                tyler@lpfutures.com
  [ 9/10] Matthew              guissetmatthew@gmail.com
  [10/10] Cono

In [3]:
# Cell 3 — Load Scraped Data
date_str = datetime.now().strftime('%Y-%m-%d')
output_file = OUTPUT_DIR / f'linkedin_connections_{date_str}.json'

if not output_file.exists():
    raise FileNotFoundError(f'Expected output file not found: {output_file.name}')

connections = json.loads(output_file.read_text(encoding='utf-8'))
print(f'Loaded {len(connections)} connection(s) from {output_file.name}')
print()

# Preview
for c in connections:
    email_display = c.get('email') or '— no email'
    print(f"  {c['first_name']:<20} {email_display}")

Loaded 10 connection(s) from linkedin_connections_2026-03-29.json

  Daria                wk021358@student.reading.ac.uk
  David                rivasentrepeneur@gmail.com
  Saad                 saadelhaddadi.5@gmail.com
  Aman                 shaikhaman0207@gmail.com
  Denis                denis.desclos@gmail.com
  Oluwaseun            kykuit_merchandise@yahoo.com
  Arnav                arnav.s2207@gmail.com
  Tyler                tyler@lpfutures.com
  Matthew              guissetmatthew@gmail.com
  Conor                14cbrient@gmail.com


## Step 2 — Deduplicate Against Existing Notion Leads

Query `leads_crm` for all existing emails and skip any connection
already present to avoid duplicates.

In [ ]:
# Cell 4 — Fetch Existing Leads & Deduplicate
# Only fetch leads that actually have an email — saves API payload & time
print('Fetching existing leads from Notion (email ≠ empty)...')
existing_leads = query_db('leads_crm', filter={
    'property': 'Email',
    'email': {'is_not_empty': True},
})
existing_emails = {
    (row.get('Email') or '').strip().lower()
    for row in existing_leads
    if row.get('Email')
}
print(f'Found {len(existing_emails)} existing email(s) in leads_crm.')

# Filter to only connections that have email AND aren't already in Notion
new_leads = [
    c for c in connections
    if c.get('email')
    and c['email'].strip().lower() not in existing_emails
]

skipped_no_email = sum(1 for c in connections if not c.get('email'))
skipped_duplicate = len(connections) - skipped_no_email - len(new_leads)

print(f'\n📊 Summary:')
print(f'  Total scraped:    {len(connections)}')
print(f'  No email:         {skipped_no_email}')
print(f'  Already in CRM:   {skipped_duplicate}')
print(f'  New to push:      {len(new_leads)}')

if new_leads:
    print(f'\nNew leads to add:')
    for c in new_leads:
        print(f"  • {c['first_name']:<20} {c['email']}")
else:
    print('\n✅ All connections already in CRM — nothing to add.')

Fetching existing leads from Notion...
Found 179 existing email(s) in leads_crm.

📊 Summary:
  Total scraped:    10
  No email:         0
  Already in CRM:   0
  New to push:      10

New leads to add:
  • Daria                wk021358@student.reading.ac.uk
  • David                rivasentrepeneur@gmail.com
  • Saad                 saadelhaddadi.5@gmail.com
  • Aman                 shaikhaman0207@gmail.com
  • Denis                denis.desclos@gmail.com
  • Oluwaseun            kykuit_merchandise@yahoo.com
  • Arnav                arnav.s2207@gmail.com
  • Tyler                tyler@lpfutures.com
  • Matthew              guissetmatthew@gmail.com
  • Conor                14cbrient@gmail.com


## Step 3 — Push New Leads to Notion

Each new lead is added to `leads_crm` with:
- **Subject** (title): full name
- **First Name**: first name
- **Email**: email address
- **Source**: `LinkedIn`

In [ ]:
# Cell 5 — Push to Notion leads_crm (batch, rate-limit aware)
if not new_leads:
    print('Nothing to push.')
else:
    rows = [
        {
            'Subject':    lead['full_name'],
            'First Name': lead['first_name'],
            'Email':      lead['email'],
            'Source':     'LinkedIn',
        }
        for lead in new_leads
    ]
    result = batch_add_rows('leads_crm', rows, label='leads')
    added  = result['added']
    errors = result['errors']

  [ 1/10] ✅ Daria                wk021358@student.reading.ac.uk  (page: 332652ea)
  [ 2/10] ✅ David                rivasentrepeneur@gmail.com  (page: 332652ea)
  [ 3/10] ✅ Saad                 saadelhaddadi.5@gmail.com  (page: 332652ea)
  [ 4/10] ✅ Aman                 shaikhaman0207@gmail.com  (page: 332652ea)
  [ 5/10] ✅ Denis                denis.desclos@gmail.com  (page: 332652ea)
  [ 6/10] ✅ Oluwaseun            kykuit_merchandise@yahoo.com  (page: 332652ea)
  [ 7/10] ✅ Arnav                arnav.s2207@gmail.com  (page: 332652ea)
  [ 8/10] ✅ Tyler                tyler@lpfutures.com  (page: 332652ea)
  [ 9/10] ✅ Matthew              guissetmatthew@gmail.com  (page: 332652ea)
  [10/10] ✅ Conor                14cbrient@gmail.com  (page: 332652ea)

✅ Added: 10  |  ❌ Failed: 0


In [6]:
# Cell 6 — Verify: query back the newly added leads
if added:
    print('Verifying newly added leads in Notion...')
    verify = query_db('leads_crm', filter={
        'property': 'Source',
        'select': {'equals': 'LinkedIn'}
    })
    linkedin_leads = [r for r in verify if r.get('Source') == 'LinkedIn']
    print(f'Total LinkedIn leads in CRM: {len(linkedin_leads)}')
    print()
    for r in linkedin_leads[:20]:
        print(f"  {r.get('First Name', ''):<20} {r.get('Email', '')}")
else:
    print('Nothing was added — skip verification.')

Verifying newly added leads in Notion...
Total LinkedIn leads in CRM: 131

  Hojun                rickhjlee@gmail.com
  Ryan                 ryanghasry84@gmail.com
  Leon                 leonkazungu5@gmail.com
  Hoang                hoangvanhieu.hvnh@gmail.com
  Christian            christianmoodie2@gmail.com
  adesh                adeshthosani76@gmail.com
  Muhammed             muhammed.olafusi@gmail.com
  Nahuel               drnahuelfinance@gmail.com
  Diego                diegogaliachoo@gmail.com
  Rene                 reneloch00@gmail.com
  German               germancerratejr@gmail.com
  Spyridon             asamoilis31@gmail.com
  Graziano             domicaba50@gmail.com
  Vasu                 vasu9167@gmail.com
  Alexis               a.tricaud85@gmail.com
  Tom                  tom@trader-dante.com
  J.                   jumadil772@gmail.com
  Tariq                tariq.skeete@hotmail.co.uk
  Qasim                qasimkhadim64@gmail.com
  Zayn                 zaynahmed0101@gma